# Phase 7 — train the two tiny heads (Colab)

What this does: keeps ConvNeXt-Tiny + DINOv2 **frozen**, trains only two
one-layer classifiers (~2k params) + one temperature number for the FFT
slope. Sends back kilobyte-size files you commit to the repo.

**Run order:** top to bottom. Two things you supply: dataset folders
(cell 1) and a Drive folder for cache (cell 2 keeps everything on
Drive so Colab timeouts cost you nothing).

## 1. Config — edit these, then run everything below

Folder convention (both roots identical):
`<root>/train/real/*.jpg`, `<root>/train/fake/*.jpg`, same for
`val/` and `test/`. Real = camera photos, fake = AI/manipulated.
Same person must never appear in two splits (split by identity
when you extract frames, not randomly).

In [ ]:
# --- Edit these three paths --- #
DATA_ROOT = "/content/drive/MyDrive/deepfake-data/ffpp-frames"  # train/val/test
EVAL_ROOT = "/content/drive/MyDrive/deepfake-data/celeba-frames"  # cross-dataset exam (same layout, test/ used)
WORK_DIR = "/content/drive/MyDrive/deepfake-data/work"  # cache + outputs (survives timeouts)

# --- Leave these alone unless training misbehaves --- #
SEED = 7
BATCH = 256
EPOCHS = 50
LR = 1e-3

print("train root:", DATA_ROOT)
print("eval root :", EVAL_ROOT)
print("work dir  :", WORK_DIR)

## 2. Setup — Drive, libraries, repo code

In [ ]:
# Mount Drive (approves one popup, keeps data + cache across sessions).
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Libraries: torch exists already (CUDA build). Add the model zoos + metrics.
!pip install -q timm transformers scikit-learn

In [ ]:
# Repo code: clone once, reuse the SAME preprocessing the pipeline uses
# (garbage in training = garbage in production if pixels differ).
import sys
from pathlib import Path

REPO = Path("/content/deepfake-detector")

if not REPO.exists():
    !git clone -q https://github.com/Ashking-tech/deepfake-detector.git $REPO

sys.path.insert(0, str(REPO))

from app.pillars.forensics import preprocess
from app.pillars.forensics import fft_branch

import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())

## 3. File lists — one row per image: (path, label, split)

Label rule: `real/` = 0, `fake/` = 1. Counts print below — eyeball
that no split is empty and classes are roughly balanced.

In [ ]:
# Collects (path, label) rows from <root>/<split>/{real,fake}/ folders.
def collect(root, splits):
    rows = []
    for split in splits:
        for label_name, label in [("real", 0), ("fake", 1)]:
            folder = Path(root) / split / label_name
            for ext in ("*.jpg", "*.jpeg", "*.png", "*.webp"):
                for path in sorted(folder.glob(ext)):
                    rows.append((str(path), label, split))
    return rows

train_rows = collect(DATA_ROOT, ["train", "val"])
test_rows = collect(DATA_ROOT, ["test"])
x_rows = collect(EVAL_ROOT, ["test"])

for name, rows in [("train+val", train_rows), ("test", test_rows), ("cross", x_rows)]:
    n_fake = sum(1 for _, label, _ in rows if label == 1)
    print(f"{name}: {len(rows)} images ({n_fake} fake)")

assert len(train_rows) > 0, "DATA_ROOT train/val is empty — check the path + layout"
assert len(test_rows) > 0, "DATA_ROOT test/ is empty"

## 4. Feature extraction — frozen backbones, run once, cached

This is the slow cell (one forward pass per image). Results save to
an `.npz` in WORK_DIR — re-running the notebook reloads the cache
instead of re-extracting. Backbones stay frozen: no gradients here.

In [ ]:
# One image file -> (convnext 768, dino 768, fft slope). Same pixels
# the production pipeline sees (preprocess.prepare, one shared path).
import numpy as np

def extract_one(path):
    # Step 1: load + shared preparation (square + 224x224).
    image = preprocess.load_image(path)
    small = preprocess.prepare(image)

    # Step 2: ConvNeXt 768-vector (frozen, eval mode, no grad).
    from app.pillars.forensics import convnext
    rgb = preprocess.to_array(small)
    conv_vec = convnext.embed(rgb)

    # Step 3: DINOv2 CLS 768-vector (frozen, eval mode, no grad).
    from app.pillars.forensics import dino_probe
    dino_vec = dino_probe.cls_embed(small)

    # Step 4: FFT spectral slope (the live heuristic signal).
    gray = rgb.mean(axis=2)
    slope = fft_branch.spectral_slope(gray)

    return conv_vec, dino_vec, slope


# Whole row-list -> cached .npz (skips work if the cache already exists).
def extract_all(rows, cache_name):
    cache_path = Path(WORK_DIR) / cache_name
    cache_path.parent.mkdir(parents=True, exist_ok=True)

    if cache_path.exists():
        print("cache hit:", cache_path)
        cached = np.load(cache_path)
        return cached["conv"], cached["dino"], cached["slope"], cached["label"]

    conv_list, dino_list, slope_list, label_list = [], [], [], []

    for index, (path, label, split) in enumerate(rows):
        conv_vec, dino_vec, slope = extract_one(path)
        conv_list.append(conv_vec)
        dino_list.append(dino_vec)
        slope_list.append(slope)
        label_list.append(label)

        if (index + 1) % 200 == 0:
            print(f"  {index + 1}/{len(rows)}")

    conv = np.stack(conv_list).astype(np.float32)
    dino = np.stack(dino_list).astype(np.float32)
    slope = np.array(slope_list, dtype=np.float32)
    label = np.array(label_list, dtype=np.float32)

    np.savez_compressed(cache_path, conv=conv, dino=dino, slope=slope, label=label)
    print("cached:", cache_path, conv.shape)
    return conv, dino, slope, label


conv_tr, dino_tr, slope_tr, y_tr = extract_all(train_rows, "features_trainval.npz")
conv_te, dino_te, slope_te, y_te = extract_all(test_rows, "features_test.npz")

if len(x_rows) > 0:
    conv_x, dino_x, slope_x, y_x = extract_all(x_rows, "features_cross.npz")
else:
    conv_x = dino_x = slope_x = y_x = None
    print("no cross-dataset frames — cross exam skipped (add EVAL_ROOT later)")

## 5. Train the two heads — one linear layer each, seconds not hours

Frozen 768-vectors in, fake-probability out. Plain logistic regression
trained with torch (BCE loss, Adam). The backbones never move.

In [ ]:
# Trains a single Linear(768 -> 1) probe on frozen features.
import torch.nn as nn

torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("training on:", device)


def train_head(features, labels):
    # Step 1: split train rows 90/10 into fit/early-stop (by row order;
    # identity separation was already enforced when disks were split).
    n = len(features)
    cut = int(n * 0.9)
    x_fit = torch.from_numpy(features[:cut]).to(device)
    y_fit = torch.from_numpy(labels[:cut]).to(device)

    # Step 2: one linear layer + sigmoid (logistic regression).
    head = nn.Linear(features.shape[1], 1).to(device)
    opt = torch.optim.Adam(head.parameters(), lr=LR)
    loss_fn = nn.BCEWithLogitsLoss()

    # Step 3: full-batch steps (features are tiny — no loader needed).
    head.train()
    for epoch in range(EPOCHS):
        opt.zero_grad()
        logits = head(x_fit).squeeze(1)
        loss = loss_fn(logits, y_fit)
        loss.backward()
        opt.step()

    # Step 4: hand back CPU weights (repo loads them without CUDA).
    head = head.cpu().eval()
    print(f"  final loss: {loss.item():.4f}")
    return head


conv_head = train_head(conv_tr, y_tr)
dino_head = train_head(dino_tr, y_tr)

## 6. Fit the FFT temperature — grid search on train AUC

`slope_to_score(slope, temperature=T)` squashes the raw slope into a
fake-probability. The repo ships `T = 1.0` (uncalibrated identity).
We try 12 candidates and keep the one with the best train AUC.

In [ ]:
# Picks the temperature with the best ranking power (AUC) on train rows.
from sklearn.metrics import roc_auc_score

CANDIDATES = [0.2, 0.4, 0.6, 0.8, 1.0, 1.25, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0]

best_t, best_auc = 1.0, -1.0

for temp in CANDIDATES:
    scores = [fft_branch.slope_to_score(s, temperature=temp) for s in slope_tr]
    auc = roc_auc_score(y_tr, scores)
    print(f"  T={temp:<5} train AUC={auc:.4f}")
    if auc > best_auc:
        best_t, best_auc = temp, auc

print("picked temperature:", best_t)

## 7. Report card — in-domain test + cross-dataset exam

AUC = ranking power (0.5 = coin flip, 1.0 = perfect). The cross row
is the honest one: generators the heads never trained on.
Ship rule of thumb: in-domain ≥ 0.90, cross ≥ 0.75, else keep the
corresponding weight at 0.0 and say so in the metrics file.

In [ ]:
# AUC of each signal on a (features, labels) pair. No training here.
from sklearn.metrics import roc_auc_score


def head_scores(head, features):
    head.eval()
    with torch.no_grad():
        logits = head(torch.from_numpy(features)).squeeze(1)
        probs = torch.sigmoid(logits).numpy()
    return probs


def report_block(title, conv_f, dino_f, slope_v, labels):
    conv_auc = roc_auc_score(labels, head_scores(conv_head, conv_f))
    dino_auc = roc_auc_score(labels, head_scores(dino_head, dino_f))
    fft_scores = [fft_branch.slope_to_score(s, temperature=best_t) for s in slope_v]
    fft_auc = roc_auc_score(labels, fft_scores)
    print(f"{title}: convnext={conv_auc:.4f} dino={dino_auc:.4f} fft={fft_auc:.4f}")
    return {"convnext_auc": float(conv_auc), "dino_auc": float(dino_auc), "fft_auc": float(fft_auc)}


metrics = {}
metrics["test"] = report_block("in-domain test", conv_te, dino_te, slope_te, y_te)

if y_x is not None:
    metrics["cross"] = report_block("cross-dataset ", conv_x, dino_x, slope_x, y_x)
else:
    metrics["cross"] = None

## 8. Ship it — save 3 small files, print hashes

Copy these from WORK_DIR back to the repo: heads go to
`weights/heads/` (gitignored — large-ish? no, kilobytes), metrics +
temperature go to `training/phase7_metrics.json` (committed). The
repo-side wiring (loading these in `convnext.py` / `dino_probe.py`
and flipping the zero-weights) happens back home, not here.

In [ ]:
# Saves heads + temperature + metrics, then prints SHA-256 fingerprints.
import hashlib
import json
from datetime import datetime, timezone

OUT = Path(WORK_DIR) / "phase7_out"
OUT.mkdir(parents=True, exist_ok=True)

# Step 1: the two trained heads (state_dicts, CPU tensors, kilobytes).
torch.save(conv_head.state_dict(), OUT / "convnext_head.pt")
torch.save(dino_head.state_dict(), OUT / "dino_head.pt")

# Step 2: temperature + full metrics + provenance of this run.
payload = {}
payload["temperature"] = best_t
payload["metrics"] = metrics
payload["seed"] = SEED
payload["epochs"] = EPOCHS
payload["lr"] = LR
payload["train_root"] = DATA_ROOT
payload["eval_root"] = EVAL_ROOT
payload["made_utc"] = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")

with open(OUT / "phase7_metrics.json", "w") as handle:
    json.dump(payload, handle, indent=2)

# Step 3: fingerprints (paste these into the repo-side commit message).
for name in ["convnext_head.pt", "dino_head.pt", "phase7_metrics.json"]:
    digest = hashlib.sha256((OUT / name).read_bytes()).hexdigest()[:16]
    size = (OUT / name).stat().st_size
    print(f"{name} ({size} bytes) sha256:{digest}")

print("copy these three files out of:", OUT)